# Panopticon Protocol v3 - Fixed Colab Training Notebook

This notebook is updated for the training/evaluation bug fixes made after the poor trained-model results. Run the cells manually from top to bottom. Do **not** use TPU for this code path.

Use this notebook with the companion hotfix archive:

`panopticon_training_eval_hotfix.zip`

If GitHub has not been updated yet, the notebook will ask you to upload that zip and will patch the cloned repo inside Colab before training.

## Runtime Choice

Use `Runtime -> Change runtime type -> T4 GPU` in free Colab.

Do **not** use `v5e-1 TPU` here. This training script uses PyTorch CUDA (`torch.cuda`) with Transformers, PEFT, and TRL. TPU training would require a separate PyTorch/XLA implementation that this project does not contain.

## 1. Install Dependencies

In [ ]:
!pip uninstall -y torchao || true
!pip install -q \
  pydantic==2.6.1 \
  gymnasium==0.29.1 \
  numpy==1.26.4 \
  transformers==4.46.3 \
  tokenizers==0.20.3 \
  trl==0.12.2 \
  peft==0.12.0 \
  accelerate==0.34.2 \
  datasets==3.1.0 \
  huggingface-hub==0.26.5 \
  safetensors==0.4.5 \
  matplotlib==3.9.2 \
  fastapi httpx tqdm

## 2. Optional Hugging Face Login

Login is only needed if you want to upload the final model/artifacts or access private repos. Training with public Qwen should work without it.

In [ ]:
import os
from getpass import getpass

HF_TOKEN = os.environ.get("HF_TOKEN", "")
DO_HF_LOGIN = False  # change to True only if you plan to upload later

if DO_HF_LOGIN and not HF_TOKEN:
    HF_TOKEN = getpass("Paste your Hugging Face token: ")

if DO_HF_LOGIN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("Logged in to Hugging Face.")
else:
    print("Skipping Hugging Face login for now.")

## 3. Clone Repository

This clones the current GitHub repo. If GitHub still has old code, the next cell patches it using the hotfix zip.

In [ ]:
!rm -rf /content/panopticon-protocol-v3
!git clone https://github.com/Ayush-Kumar0207/panopticon-protocol-v3.git /content/panopticon-protocol-v3
%cd /content/panopticon-protocol-v3

from pathlib import Path
required = ["environment.py", "models.py", "grader.py", "train_trl_v2.py", "inference_local.py"]
for file in required:
    print(("OK" if Path(file).exists() else "MISSING"), file)

## 4. Apply Hotfix If GitHub Has Old Code

Run this cell. If it prints any missing fix marker, upload `panopticon_training_eval_hotfix.zip` when prompted.

In [ ]:
%cd /content/panopticon-protocol-v3

from pathlib import Path
import zipfile

FIX_CHECKS = {
    "train_trl_v2.py": [
        "curriculum-expert-v4-compact",
        "DataCollatorForCompletionOnlyLM",
        "Allowed Departments",
    ],
    "inference_local.py": [
        "_priority_repair_action",
    ],
    "models.py": [
        "_observable_departments",
    ],
}

def check_fixed(verbose=True):
    all_ok = True
    for file, needles in FIX_CHECKS.items():
        text = Path(file).read_text(encoding="utf-8")
        for needle in needles:
            ok = needle in text
            if verbose:
                print(("OK" if ok else "MISSING"), file, needle)
            all_ok = all_ok and ok
    return all_ok

if not check_fixed(verbose=True):
    print("\nGitHub clone has old code. Upload panopticon_training_eval_hotfix.zip now.")
    from google.colab import files
    uploaded = files.upload()
    zip_candidates = [name for name in uploaded if name.endswith(".zip")]
    if not zip_candidates:
        raise RuntimeError("No zip uploaded. Upload panopticon_training_eval_hotfix.zip and rerun this cell.")
    zip_path = Path(zip_candidates[0])
    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall("/content/panopticon-protocol-v3")
    print(f"Applied hotfix from {zip_path}.")

print("\nFinal fixed-code check:")
if not check_fixed(verbose=True):
    raise RuntimeError("STOP: fixed code is still missing. Do not train yet.")
print("Fixed code is present. Safe to continue.")

## 5. Smoke Test

This verifies the environment and grader after the hotfix.

In [ ]:
%cd /content/panopticon-protocol-v3
!python smoke_test.py

## 6. Configure T4 Training Run

Recommended on free Colab T4:

- `EPISODES_PER_LEVEL = 20` for a practical fixed run.
- Change to `50` only if you have a long stable GPU session and enough time.

The old bad run had trained `sleepers_caught_mean = 0.0`; even a 20-episode fixed run should be enough to confirm the bug is gone.

Training now writes both `output_logs.txt` and a structured Drive log named `training_events.jsonl`; the plot cell reads those logs directly.

In [ ]:
import os
import torch
from google.colab import drive

drive.mount("/content/drive")

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("No CUDA GPU detected. Switch runtime to T4 GPU, not TPU.")

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
EPISODES_PER_LEVEL = 20  # T4 practical default. Set to 50 for final-quality run if time allows.
EVAL_EPISODES = 3        # Set to 5 for final reporting after a successful run.

RUN_NAME = f"panopticon-fixed-v3-ep{EPISODES_PER_LEVEL}"
os.environ["TRAIN_ROOT"] = f"/content/drive/MyDrive/{RUN_NAME}"
TRAIN_ROOT = os.environ["TRAIN_ROOT"]
TRAIN_EVENTS_PATH = os.path.join(TRAIN_ROOT, "training_events.jsonl")
os.environ["TRAIN_EVENTS_PATH"] = TRAIN_EVENTS_PATH
MERGED_MODEL_PATH = os.path.join(TRAIN_ROOT, "merged_model")
EVAL_PLOT_DIR = f"plots/evaluation_fixed_ep{EPISODES_PER_LEVEL}"

TRAIN_ARGS = f"--curriculum --model {MODEL_NAME} --episodes {EPISODES_PER_LEVEL} --merge"
DO_UPLOAD = False

print(f"TRAIN_ARGS={TRAIN_ARGS}")
print(f"EVAL_EPISODES={EVAL_EPISODES}")
print(f"TRAIN_ROOT={TRAIN_ROOT}")
print(f"TRAIN_EVENTS_PATH={TRAIN_EVENTS_PATH}")
print(f"MERGED_MODEL_PATH={MERGED_MODEL_PATH}")
print(f"EVAL_PLOT_DIR={EVAL_PLOT_DIR}")
print(f"DO_UPLOAD={DO_UPLOAD}")

## 7. Train Fixed Curriculum Model

This is the long cell. It saves data/model under Google Drive so an interrupted run can resume.

Watch for `[DATA_PROGRESS]` during expert data generation and `[TRAIN_PROGRESS]` during SFT. Detailed per-turn/action/reward telemetry is saved to `TRAIN_EVENTS_PATH` for plotting.

If this cell finishes in only one or two minutes, training did not run; inspect `output_logs.txt` and fix the displayed Python error before continuing.

In [ ]:
%cd /content/panopticon-protocol-v3

import subprocess

train_cmd = f"set -o pipefail; python -u train_trl_v2.py {TRAIN_ARGS} 2>&1 | tee output_logs.txt"
print(train_cmd)
subprocess.run(["bash", "-lc", train_cmd], check=True)
print(f"Structured event log saved at: {TRAIN_EVENTS_PATH}")

## 8. Generate Training Plots And Evaluate

This creates training plots from `output_logs.txt` plus `training_events.jsonl`, then runs the benchmark on the merged model.

In [ ]:
%cd /content/panopticon-protocol-v3

import subprocess
from pathlib import Path

subprocess.run("python generate_plots.py", shell=True, check=True)
print(f"Event log exists: {Path(TRAIN_EVENTS_PATH).exists()} -> {TRAIN_EVENTS_PATH}")

print(f"Using merged model from: {MERGED_MODEL_PATH}")
print(f"Exists: {Path(MERGED_MODEL_PATH).exists()}")
if not Path(MERGED_MODEL_PATH).exists():
    raise FileNotFoundError("Merged model not found. Make sure training finished and used --merge.")

eval_cmd = (
    f"python full_evaluation.py "
    f"--model {MERGED_MODEL_PATH} "
    f"--episodes {EVAL_EPISODES} "
    f"--output evaluationResults_fixed.json "
    f"--plot-dir {EVAL_PLOT_DIR} "
    f"--showcase-output showcaseResults_fixed.json"
)
print(eval_cmd)
result = subprocess.run(eval_cmd, shell=True, text=True, capture_output=True)
if result.stdout:
    print(result.stdout)
if result.stderr:
    print(result.stderr)
result.check_returncode()

## 9. Inspect The Key Metric

The old bad trained model caught zero sleepers. This cell checks whether the fixed run actually catches sleepers.

In [ ]:
import json
from pathlib import Path

payload_path = Path("evaluationResults_fixed.json")
if not payload_path.exists():
    raise FileNotFoundError("Run the evaluation cell first.")

payload = json.loads(payload_path.read_text(encoding="utf-8"))
print("Trained model summary:")
all_zero = True
for level, row in payload["agents"]["trained"]["summary"].items():
    caught = row["sleepers_caught_mean"]
    all_zero = all_zero and caught == 0
    print(level, "grade=", round(row["grade_mean"], 3), "caught=", caught, "security=", round(row["security_mean"], 1))

if all_zero:
    raise RuntimeError("The trained model still caught zero sleepers. Do not upload this result.")
print("Fixed run is not the old zero-catch failure mode.")

## 10. Optional Upload To Hugging Face

Leave `DO_UPLOAD = False` until the metrics look good. If you later set it to `True`, this uploads the merged model and fixed artifacts.

In [ ]:
from pathlib import Path

if not DO_UPLOAD:
    print("DO_UPLOAD=False, skipping upload. Inspect metrics first.")
else:
    from huggingface_hub import HfApi
    if not HF_TOKEN:
        raise RuntimeError("Set HF_TOKEN / run Hugging Face login before uploading.")

    api = HfApi(token=HF_TOKEN)
    repo_id = "Ayush-Kumar0207/panopticon-argus-qwen-1.5B-fixed"
    api.create_repo(repo_id, repo_type="model", exist_ok=True)

    api.upload_folder(folder_path=MERGED_MODEL_PATH, repo_id=repo_id, repo_type="model")
    for filename in ["evaluationResults_fixed.json", "showcaseResults_fixed.json", "output_logs.txt"]:
        path = Path(filename)
        if path.exists():
            api.upload_file(path_or_fileobj=str(path), path_in_repo=filename, repo_id=repo_id, repo_type="model")
    events_path = Path(TRAIN_EVENTS_PATH)
    if events_path.exists():
        api.upload_file(path_or_fileobj=str(events_path), path_in_repo="training_events.jsonl", repo_id=repo_id, repo_type="model")
    if Path("plots").exists():
        api.upload_folder(folder_path="plots", path_in_repo="plots", repo_id=repo_id, repo_type="model")
    print(f"Uploaded fixed artifacts to https://huggingface.co/{repo_id}")